# Lesson 17 · Eigendecomposition A = PDP⁻¹ — find a matrix's "natural coordinate system" and matrix multiplication collapses into scalar multiplication

**Learning goals**
1. Write the characteristic equation `det(A − λI) = 0` and compute 2×2 eigenvalues by hand
2. Understand diagonalization `A = P D P⁻¹`: once you switch to the eigenbasis, multiplication becomes scalar multiplication
3. Know how to use `Aⁿ = P Dⁿ P⁻¹` to compute matrix powers
4. Know that **diagonalization isn't always possible**: without n independent eigenvectors you can't fill out P (this lesson touches on where that boundary lies)

> L14 already showed you `Av=λv`. This section **packages "one direction" into a decomposition of the whole matrix**; symmetric matrices are the friendly case that can almost always be diagonalized.

**Why it matters for Aurora**: the stability of a recursive filter or a state space model is decided by the eigenvalues of its system matrix — if every `|λ| < 1`, the filter converges. The pole analysis in `aurora.audio.transforms` relies on this decomposition.

← **Previous lesson** · [L16 · Determinants and inverses](L16_determinant_inverse.ipynb)

> Last lesson covered **determinants and inverses**: the geometric meaning of det(A) (area scaling), inverting a matrix, and when you can't.
> This lesson explores **eigendecomposition A=PDP⁻¹**.

## This lesson's story: change the coordinate system, and multiplication turns into plain scalar multiplication

Have you ever wondered why some calculations are especially easy in one particular coordinate system?

An ordinary matrix product `A @ v` is hard work — n² multiply-adds every time. But if `v` is an eigenvector, the product collapses into **scalar multiplication**: `A @ v = λ · v`, a single number times a vector.

Eigendecomposition finds a "natural coordinate system" in which the matrix becomes a diagonal matrix D:

```
A = P D P⁻¹
```

What it means:
- **P**: switch into the eigenvector coordinate system
- **D**: in the new coordinates, all A does is scale each direction independently (diagonal entries = eigenvalues)
- **P⁻¹**: switch back to the original coordinate system

What is the decomposition good for?
- **Matrix powers**: `Aⁿ = P Dⁿ P⁻¹`, and Dⁿ only needs each diagonal entry raised to the n-th power
- **Stability analysis**: if every |λᵢ| < 1, then Aⁿ → 0 (the system converges)
- **PCA**: the eigenvectors of the covariance matrix = the directions in which the data varies most

In this lesson you'll implement `char_poly(A, λ) = det(A - λI)` and check that the true eigenvalues make it zero.

## Digging deeper: how can "one eigenvector" hold up "a whole new coordinate system"?

`Av = λv` only describes the fate of **one** vector v — A stretches or flips it by a factor of λ. But above we said "the whole matrix A can be written as P D P⁻¹". That jumps from a relation about one vector to a decomposition of the entire matrix, and a step is missing in between. Let's fill it in.

**An everyday analogy**: when you move house, if the wardrobe slides into the room straight along a wall (no turning), that direction is the wardrobe's "easy direction". But a room usually holds several pieces of furniture, and each one may have its own easy direction (the bookshelf along another wall, the bed along a third). If you **take those easy directions as new coordinate axes** (bookshelf direction as the x-axis, bed direction as the y-axis, …), then describing "moving the furniture" in this new coordinate system makes every direction independent — none gets in the others' way. That is exactly what eigendecomposition does: find the directions a matrix "likes" and use them as the new axes.

**From one relation to a whole matrix equation**: suppose we have already found n eigenvectors v₁, v₂, …, vₙ (with eigenvalues λ₁, λ₂, …, λₙ). Stack them as columns of a matrix:

```
P = [ v₁ | v₂ | ... | vₙ ]
```

Now multiply A onto P. Multiplying a matrix by a set of column vectors is the same as multiplying A onto each column separately (the matrix multiplication rule from L12):

```
A P = A [v₁|v₂|...|vₙ] = [Av₁ | Av₂ | ... | Avₙ] = [λ₁v₁ | λ₂v₂ | ... | λₙvₙ]
```

And `[λ₁v₁ | λ₂v₂ | ... | λₙvₙ]` is exactly `P` times a diagonal matrix `D = diag(λ₁,...,λₙ)` — because "multiplying by a diagonal matrix on the right" just scales each column by the matching diagonal entry (the rule from L13). So:

```
A P = P D
```

If P is invertible (that is, the n eigenvectors are linearly independent and make up a full set of axes — "Common misconceptions" below explains when they can't), multiply both sides on the right by P⁻¹:

```
A = P D P⁻¹
```

Done. No magic: we took n relations of the form "one vector gets scaled on its own", **packaged them into one matrix equation** using the rules of matrix multiplication, and solved for A. The code below lets you see for yourself that the intermediate equation `AP = PD` really holds.


In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
# Check the key bridge in the derivation step by step: AP = PD (this step is more basic than the final A=PDP⁻¹)
import numpy as np

A = np.array([[4., 1.], [2., 3.]])
lam, P = np.linalg.eig(A)          # each column of P is an eigenvector
D = np.diag(lam)

AP = A @ P                          # left side: A acting on each of the two eigenvectors
PD = P @ D                          # right side: each column of P scaled by its eigenvalue

print('A @ P =\n', np.round(AP, 4))
print('P @ D =\n', np.round(PD, 4))
print('Are they equal?', np.allclose(AP, PD))


## 1. The characteristic equation: `det(A − λI) = 0`

**How do you find eigenvalues?**

`Av = λv` is equivalent to `(A - λI)v = 0`, which has a nonzero solution v if and only if `A - λI` is singular:

$$\det(A - \lambda I) = 0$$

The roots of this equation are the eigenvalues.

**A 2×2 example by hand**: A = [[4, 1], [2, 3]]

```
det(A - λI) = (4-λ)(3-λ) - 2·1
            = λ² - 7λ + 12 - 2
            = λ² - 7λ + 10
            = (λ-5)(λ-2) = 0
```

Eigenvalues: λ₁ = 5, λ₂ = 2.

**Check in Python**: `np.linalg.eig(A)` returns the eigenvalues directly, and `char_poly(A, 5)` should be approximately 0.

> An n×n matrix has n eigenvalues (counting repeats and complex ones); a real symmetric matrix has only real eigenvalues.

### Digging deeper: why does "has a nonzero solution" mean "the determinant is 0"?

This is the step in the course where people most often get stuck, so let's take it apart slowly.

**Analogy**: think of the matrix M as a meat grinder. You feed a vector x in one end, and Mx comes out the other. Normally, different inputs come out as different outputs — the machine is "invertible", and in principle you could work back from what came out to what went in (that's what the inverse matrix M⁻¹ does).

But suppose the grinder has a "design flaw": you feed in some ingredient v that is **not all zeros**, and what comes out is **empty** (the zero vector). That means the machine squashes different inputs onto the same output — it has lost information and can't be run backwards. In that case we say M is **singular**, i.e. not invertible.

**What does this have to do with the determinant?** The determinant (from L16) measures how much a matrix scales the "volume" of space. If the matrix is singular — it squashes some nonzero vector down to 0 — then it flattens the whole space by at least one dimension (a cube pressed into a sheet of paper, or even into a line). The flattened volume is 0, so the determinant is 0. Conversely, if det(M) ≠ 0, the volume hasn't been flattened to 0, the matrix keeps information in every dimension and is invertible, so Mx=0 has only the solution x=0 (the zero vector going in and the zero vector coming out loses nothing).

These are three ways of saying the same thing:
1. `(A-λI)v = 0` has a nonzero solution v
2. `A-λI` is a singular (non-invertible) matrix
3. `det(A-λI) = 0`

So "finding the λ that really makes Av=λv hold" is the same as "finding the λ that turns A-λI into a lossy grinder (singular)", which means solving `det(A-λI)=0`. The code below checks all three on concrete numbers: at a true eigenvalue the rank drops and det becomes 0; plug in a number that isn't an eigenvalue and neither happens.


In [ ]:
# See for yourself: at a true eigenvalue the matrix is "singular" (rank drops, det=0); elsewhere it isn't
import numpy as np

A = np.array([[4., 1.], [2., 3.]])

for lam in (5, 2, 3):   # 5 and 2 are the true eigenvalues; 3 is just an arbitrary number for comparison
    M = A - lam * np.eye(2)
    rank = np.linalg.matrix_rank(M)
    det = np.linalg.det(M)
    tag = '✅ true eigenvalue (singular, rank drops)' if abs(det) < 1e-8 else '❌ not an eigenvalue (invertible, full rank)'
    print(f'λ={lam}:  rank(A-λI)={rank}  det(A-λI)={det:+.4f}  {tag}')


## ⚠️ Common misconceptions

> Don't assume that "every square matrix splits cleanly into `P D P⁻¹`". It **comes with conditions**, and these three are where beginners most often slip up:

- **Eigenvectors aren't unique**: if `v` is an eigenvector, so are `2v` and `-v` — the **direction** is what matters; the length is yours to scale. So the "length 1" vectors that `np.linalg.eig` returns are just one representative among infinitely many valid answers.
- **Not every square matrix can be diagonalized**: a matrix like `[[1,1],[0,1]]` can't muster n independent directions, so P isn't invertible and `A = PDP⁻¹` simply can't be written down (which is exactly why the SVD from L14 exists — it works for **any** matrix).
- **Eigenvalues can be complex**: a rotation matrix leaves no real direction unchanged, so its eigenvalues are a complex-conjugate pair — if you see complex numbers, don't assume you made a mistake. **Good news**: a real symmetric matrix can always be diagonalized and has only real eigenvalues, which is why covariance and Gram matrices are so pleasant to work with in Aurora.


### Extra worked example: work out by hand why `[[1,1],[0,1]]` can't fill out a set of axes

Saying such a matrix "can't muster n independent directions" is rather abstract, so let's plug in the numbers and see exactly where it breaks.

**Step 1: find the eigenvalues.** A = [[1,1],[0,1]], characteristic equation:

```
det(A-λI) = (1-λ)(1-λ) - 1·0 = (1-λ)² = 0
```

There's only one solution, λ=1, but it's a "double root" — it appears twice in the equation (this is called **algebraic multiplicity = 2**).

**Step 2: find the eigenvectors.** Substitute λ=1 and solve `(A-I)v = 0`:

```
A - I = [[0, 1], [0, 0]]
```

This system gives only one constraint: the first row says `0·v₁ + 1·v₂ = 0`, i.e. `v₂ = 0`; the first component `v₁` can be anything, and the second row `0=0` adds nothing. So the only valid eigenvectors lie on the line through `(1, 0)` (and its multiples) — **just 1 independent direction** (this is called **geometric multiplicity = 1**).

**Where's the problem?** The eigenvalue λ=1 appears twice algebraically, but only 1 independent eigen-direction goes with it. To build the 2×2 `P = [v₁|v₂]` you need 2 linearly independent columns, and here there is only 1 direction — P is a column short, can't be an invertible square matrix, and `A = PDP⁻¹` can't be written.


In [ ]:
# See for yourself: the two "eigenvectors" numpy computes actually sit on the same line (linearly dependent)
import numpy as np

A_shear = np.array([[1., 1.], [0., 1.]])
vals, vecs = np.linalg.eig(A_shear)
print('eigenvalues:', vals, ' (should be 1, 1 — the same value twice)')
print('eigenvector matrix P:\n', np.round(vecs, 4))
print('Are the two columns of P linearly independent (det(P) != 0)?', not np.isclose(np.linalg.det(vecs), 0))


### Extra worked example: why a rotation matrix gives complex eigenvalues

"Common misconceptions" said that "a rotation matrix's eigenvalues are a complex-conjugate pair". Let's plug in a concrete rotation matrix and watch the complex number pop out of the determinant equation.

**The 45° rotation matrix** (turns every vector in the plane 45° about the origin):

```
R = [[cos45°, -sin45°], [sin45°, cos45°]] ≈ [[0.707, -0.707], [0.707, 0.707]]
```

**Characteristic equation**:

```
det(R-λI) = (0.707-λ)² + 0.707²
          = λ² - 1.414λ + 1 = 0
```

Using the quadratic formula, the discriminant is `Δ = 1.414² - 4×1 = 2 - 4 = -2` — **the discriminant is negative**! High-school algebra tells us that a negative discriminant means no real solutions — but there are still two solutions, they're just complex conjugates: `λ = 0.707 ± 0.707i`.

**Geometric intuition**: an eigenvector is a vector whose "direction stays put; it only gets stretched". But a rotation turns **every** direction in the plane — unless the angle is 0° or 180°, no real direction still points the same way after rotating. Since no such direction exists in the "real plane", the equation can only give solutions in the complex numbers. That isn't a mistake; it's the geometry itself.


In [ ]:
# See for yourself: a rotation matrix's eigenvalues really are a complex-conjugate pair
import numpy as np

theta = np.pi / 4  # 45°
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
vals = np.linalg.eigvals(R)
print('rotation matrix R =\n', np.round(R, 4))
print('eigenvalues:', vals, ' (a complex-conjugate pair: same real part, opposite imaginary parts)')


### Extra proof: why a symmetric matrix's eigenvalues are guaranteed to be real

The rotation matrix above isn't symmetric (`Aᵀ ≠ A`), and it produced complex eigenvalues. So why does a symmetric matrix (`Aᵀ = A`) never run into this? Let's prove it for a general 2×2 symmetric matrix (the rigorous n×n proof needs properties of complex conjugates and is beyond this course, but the 2×2 case already shows the heart of it).

Let the symmetric matrix be `A = [[a, b], [b, d]]` (b in both the lower-left and upper-right corners — that's what "symmetric" means). Characteristic equation:

```
det(A-λI) = (a-λ)(d-λ) - b² = λ² - (a+d)λ + (ad-b²) = 0
```

By the quadratic formula, the discriminant is `Δ = (a+d)² - 4(ad-b²)`. Expand and simplify:

```
Δ = a² + 2ad + d² - 4ad + 4b²
  = a² - 2ad + d² + 4b²
  = (a-d)² + 4b²
```

Note that `(a-d)²` is a square, so it's always ≥ 0; `4b²` is a square times a positive number, also always ≥ 0. **Add two non-negative numbers and Δ is always ≥ 0** — the discriminant can never be negative, so the eigenvalues must be real, and complex numbers can't pop up the way they did for the rotation matrix. That's why the covariance and Gram matrices used in Aurora (both symmetric) can safely be assumed to have real eigenvalues, with no complex numbers to handle.


## Notation first: look at the shapes, then the operations

A vector is `(n,)`, a matrix is `(n, n)`; `A @ v` keeps the shape `(n,)`. The relation to check in this section is `A @ v == λ * v`: the left side is a matrix product, the right side is scalar scaling, and the two should match element by element.

In [ ]:
import numpy as np
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
vals, vecs = np.linalg.eig(A)
print('eigenvalues:', sorted(np.round(vals).astype(int)), ' (course notes: -3,-2,6)')

## Hands-on: eigenvectors satisfy `A @ v ≈ λ * v`

Run the code below and compare `A @ v` with `λ * v` component by component. Notice that when an eigenvalue is negative, the eigenvector gets flipped; when its absolute value is greater than 1, the vector gets stretched.

In [ ]:
# Check Av = λv: loop over the eigenvectors of a 3×3 matrix and compute the residual
import numpy as np
A_sym = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
vals, vecs = np.linalg.eig(A_sym)
print(f'{"eigenvalue λ":>12}  {"residual ‖Av−λv‖":>16}')
for lam, v in zip(vals, vecs.T):
    residual = np.linalg.norm(A_sym @ v - lam * v)
    print(f'λ = {lam:+.4f}  {residual:>16.2e}  {"✅" if residual < 1e-10 else "❌"}')
assert all(
    np.linalg.norm(A_sym @ vecs[:, i] - vals[i] * vecs[:, i]) < 1e-10
    for i in range(len(vals))
), "Av=λv check failed: inspect the eigenvector computation"
print('✅ Every eigenvector satisfies Av = λv (residual < 1e-10)')


In [ ]:
import numpy as np

# Check the eigendecomposition: A = P D P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
D = np.diag(lam)
A_rec = P @ D @ np.linalg.inv(P)
print(f'eigenvalues λ = {np.round(lam, 4)}')
print(f'A =\n{A}')
print(f'P@D@P⁻¹ =\n{np.round(A_rec, 10)}')
print(f'reconstruction error = {np.max(np.abs(A-A_rec)):.2e}')


## Code experiment: checking matrix powers `Aⁿ = P Dⁿ P⁻¹`

Compute matrix powers with the diagonalization formula `Aⁿ = P Dⁿ P⁻¹`, compare against `np.linalg.matrix_power`, and confirm the two routes agree to within machine precision.

In [ ]:
import numpy as np

# Matrix powers via diagonalization: A^n = P D^n P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
Pinv = np.linalg.inv(P)
for n in [2, 5, 10]:
    A_n_diag = P @ np.diag(lam**n) @ Pinv      # diagonalization route: D^n costs only O(N)
    A_n_iter = np.linalg.matrix_power(A, n)     # direct route (reference)
    err = np.max(np.abs(A_n_diag - A_n_iter))
    print(f'A^{n:2d}[0,0] = {A_n_diag[0,0]:.2f}  error = {err:.2e}')


## 2. ✏️ Implement the characteristic polynomial value `char_poly(A, lam) = det(A − λI)`

At an eigenvalue it should be ≈0.

**Line of reasoning**:
1. Build the shifted matrix `A - lam * np.eye(len(A))`, subtracting λ from every diagonal entry of A
2. At a true eigenvalue λ, this matrix maps the eigenvector v to the zero vector, so the matrix is singular
3. A singular matrix has determinant exactly 0; away from an eigenvalue the determinant is clearly nonzero (about 56 at λ=5, for example)

**Reference input and output**: with A as the 3×3 matrix from the course notes, `char_poly(A, -3)` ≈ 0 and `char_poly(A, 5)` ≈ +56.

> 💡 Need a hint? Once you've finished the exercise, the reference implementation is in the `solutions/` folder.


Before writing `char_poly`, be clear on three things:
- Input: a matrix `A` (n×n) and a candidate eigenvalue `lam` (a scalar)
- Key step: build `A - lam * np.eye(n)` and take its determinant
- Return: a scalar, which should be close to 0 at a true eigenvalue

In [ ]:
def char_poly(A, lam):
    import numpy as np  # guard: so np is found even if this cell is run on its own
    A = np.asarray(A, float)
    # ✏️ TODO: return det(A - lam*I)
    raise NotImplementedError("Implement char_poly: return det(A - lam * np.eye(len(A)))")


In [ ]:
# Restore the 3×3 matrix (earlier cells redefined A as a 2×2 matrix)
import numpy as np
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)

try:
    for lam in (-3, -2, 6):
        print(f'char_poly(A, {lam}) =', round(char_poly(A, lam), 6))
    assert all(abs(char_poly(A, lam)) < 1e-6 for lam in (-3,-2,6)), \
        "char_poly should be 0 at the eigenvalues — check your det implementation"
    assert abs(char_poly(A, 5) - 56) < 1.0, \
        f"char_poly(A, 5) should be ≈ +56, got {char_poly(A, 5):.2f}"
    print('✅ The characteristic polynomial is ≈0 at all three eigenvalues and +56 at a non-eigenvalue, matching the course notes.')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ Exercise not implemented yet: {e}')


## 3. Diagonalizing a symmetric matrix: `PᵀAP = D` (extra worked example)

Stack the **normalized** eigenvectors as columns of an orthogonal matrix P; then `PᵀAP` becomes a diagonal matrix, and its diagonal is exactly those eigenvalues.

Eigenvectors from the course notes: $v_1=(1,-1,-1),\ v_2=(0,1,-1),\ v_3=(2,1,1)$.

### What is an orthogonal matrix, and why can a symmetric matrix use one?

**In plain words**: an "orthogonal" matrix is one whose columns are all vectors of length 1 (called **unit vectors**), and any two columns are perpendicular (at 90°, dot product 0). Such a set of column vectors is called an **orthonormal basis** — picture a neatly arranged set of new coordinate axes that don't interfere with each other, each of length 1, just like the mutually perpendicular x/y/z axes in real life, only pointing a different way.

**How does `PᵀP=I` express "orthogonal"?** Entry (i,j) of the matrix product `PᵀP` is exactly the dot product of column i and column j of P. If every column is a unit vector (its dot product with itself = 1) and every pair is perpendicular (dot product of different columns = 0), then `PᵀP` comes out with 1s on the diagonal and 0s everywhere else — which is precisely the identity matrix `I`.

**Why is this useful to us?** For a general (non-symmetric) matrix, the `P⁻¹` in `A = PDP⁻¹` has to be computed as a separate inverse (from L16 — not cheap, and possibly numerically unstable). But if P happens to be orthogonal, `PᵀP=I` tells us directly that **Pᵀ is the inverse of P** (`P⁻¹ = Pᵀ`)! Transposing just flips the matrix across its diagonal, which costs almost nothing — far faster and more stable than inverting. That's why the diagonalization of a symmetric matrix is written `PᵀAP=D` (with Pᵀ) rather than `PDP⁻¹` (with P⁻¹) — the two forms say the same thing; in the special case of a symmetric matrix, Pᵀ and P⁻¹ just happen to be equal, so it's simpler to write.

**But why are a symmetric matrix's eigenvectors perpendicular to each other?** It isn't a coincidence; it can be proved. Let v₁ and v₂ be two eigenvectors of a symmetric matrix A with different eigenvalues λ₁≠λ₂, and consider the number `v₁ᵀ A v₂`. You can compute it two ways:

- One way: `v₁ᵀ A v₂ = v₁ᵀ(λ₂v₂) = λ₂ (v₁·v₂)` (because `Av₂=λ₂v₂`)
- The other way: `v₁ᵀ A v₂ = (Av₁)ᵀ v₂ = (λ₁v₁)ᵀ v₂ = λ₁ (v₁·v₂)` (this step uses `Aᵀ=A`, so `v₁ᵀA = (Av₁)ᵀ`)

Both compute the same number, so `λ₂(v₁·v₂) = λ₁(v₁·v₂)`, which rearranges to `(λ₁-λ₂)(v₁·v₂) = 0`. Since λ₁≠λ₂, the only option is **v₁·v₂ = 0** — the two eigenvectors have dot product zero, i.e. they're perpendicular. That's why a symmetric matrix's eigenvectors are naturally pairwise orthogonal; the code below checks it with the three eigenvectors from the course notes.


In [ ]:
# See for yourself: the three eigenvectors have pairwise dot products of 0 (mutually perpendicular)
import numpy as np

v1 = np.array([1, -1, -1.]); v2 = np.array([0, 1, -1.]); v3 = np.array([2, 1, 1.])
print('v1·v2 =', v1 @ v2)
print('v1·v3 =', v1 @ v3)
print('v2·v3 =', v2 @ v3)
print('All three dot products are 0, so they are pairwise perpendicular — which is why they can form an orthogonal matrix P.')


In [ ]:
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
v1=np.array([1,-1,-1.]); v2=np.array([0,1,-1.]); v3=np.array([2,1,1.])
P = np.column_stack([v/np.linalg.norm(v) for v in (v1,v2,v3)])
D = P.T @ A @ P
print('P orthogonal (PᵀP=I):', np.allclose(P.T@P, np.eye(3)))
print('diagonal of PᵀAP:', np.round(np.diag(D)).astype(int), '(course notes: -3,-2,6)')
assert np.allclose(D - np.diag(np.diag(D)), 0, atol=1e-9)
print('✅ Diagonalized successfully, matching the course notes.')

**🔗 Aurora connection**: the recursive-filter stability check in `aurora.audio.transforms` calls `np.linalg.eig` on the system matrix and confirms every `|λ| < 1`. PCA applies the same eigendecomposition to a covariance matrix, and it's a close cousin of the SVD from L14 — `AᵀA = VΣ²Vᵀ` is the eigendecomposition of `AᵀA`, whose eigenvalues are exactly the squares of the singular values.

**Extra worked examples covered**: the characteristic equation, properties of symmetric matrices, and orthogonal diagonalization PᵀAP=D.

### What do these terms actually mean? (for anyone who hasn't met signal processing yet)

- **Recursive filter**: a way of computing that "uses past outputs to produce the present output". The simplest example: `y[n] = a·y[n-1] + x[n]` — the output now is the previous output times a coefficient a, plus the new input now. If |a|<1, the influence of the past fades round after round and the output doesn't blow up; that's "convergence". More complex filters depend on several past moments at once; then you bundle all the past states into one vector and use a matrix A to describe "next state = A × current state" in one go — which is exactly the state space model below.

- **State space model**: bundle the system's "memory" (for instance the past values stored in a filter) into a vector `s`, and describe how it evolves over time with `s[n+1] = A @ s[n]`. That's exactly the form that keeps coming up in this lesson: a matrix A acting on a vector over and over. With the eigendecomposition `A=PDP⁻¹` we know `Aⁿ = PDⁿP⁻¹`, and the diagonal of `Dⁿ` is each eigenvalue raised to the n-th power — as long as every |λ|<1, these powers all decay to 0 and the state eventually goes to zero (convergence); if even one |λ|>1, the matching direction blows up exponentially (divergence). That's what "pole analysis" does: the "poles" here are the eigenvalues of the system matrix.

- **Covariance matrix**: measures how much the features (dimensions) of a dataset "vary together". For example, height and weight usually rise and fall together, so their covariance is positive. Bundle this relationship for every pair of dimensions into a matrix and you have the covariance matrix — it's always symmetric, so this lesson's "a symmetric matrix can be orthogonally diagonalized" applies directly. PCA (principal component analysis) is the eigendecomposition of the covariance matrix: the direction with the largest eigenvalue is the one in which the data varies most (carries the most information).


## A side note: `np.linalg.eig` or `np.linalg.eigh` — which one?

The code earlier in this lesson has used `np.linalg.eig` throughout (it works on any matrix), but the code coming up switches to `np.linalg.eigh` — with an extra 'h'. The difference:

- **`eig`**: works on any square matrix, symmetric or not. Because a non-symmetric matrix can have complex eigenvalues (like the rotation matrix earlier), `eig` leaves room for complex numbers in its result type; even when the answers are really all real, floating-point error can leave a tiny but nonzero imaginary part (say `1e-16j`), so you need a little extra care.

- **`eigh`** ('h' stands for Hermitian; a symmetric matrix is the real-valued version): made specifically for symmetric matrices. Since we've already shown that a symmetric matrix's eigenvalues are guaranteed real and its eigenvectors guaranteed pairwise orthogonal, `eigh` exploits both facts to take a faster, more numerically stable route, returning sorted real eigenvalues and a naturally orthogonal eigenvector matrix directly, with no stray imaginary parts to worry about.

**Rule of thumb**: whenever you know for sure the matrix is symmetric (a covariance matrix, a Gram matrix), prefer `eigh`; if you're not sure, or the matrix isn't symmetric to begin with, use the general `eig`.


## 🎨 Illustrated: orthogonal diagonalization of a symmetric matrix S = QΛQᵀ (extra worked example)


In [ ]:
import numpy as np
from laviz import style, show_factorization
style()
S=np.array([[3.,3,3],[3,-1,1],[3,1,-1]]); w,Q=np.linalg.eigh(S)
show_factorization(S,[Q,np.diag(w),Q.T],['Q','Λ','Qᵀ'],
                   modes=['#2A9D8F','#E9C46A','#2A9D8F'], title='S = Q Λ Qᵀ');

In [ ]:
import numpy as np

# Parameter experiment: change a matrix entry and watch the eigenvalues move
print(f"{'param c':>8}  {'λ₁':>10}  {'λ₂':>10}  {'|λ₁/λ₂|':>10}")
for c in [0.5, 1.0, 2.0, 4.0]:
    A = np.array([[3., c],[1., 2.]])
    lam = np.linalg.eigvals(A)
    lam = sorted(lam, key=abs, reverse=True)
    ratio = abs(lam[0]/lam[1]) if abs(lam[1]) > 1e-10 else float('inf')
    print(f'{c:>8.1f}  {lam[0]:>+10.4f}  {lam[1]:>+10.4f}  {ratio:>10.4f}')
print('→ The further apart the two eigenvalues, the faster power iteration converges (the dominant direction stands out more).')


### What does the ratio `|λ₁/λ₂|` actually represent?

The ratio printed above measures "how much harder the matrix stretches along its strongest direction than along the next strongest". The bigger the ratio, the more "lopsided" the matrix's action — nearly all of the stretching is concentrated in one direction, and the other directions are barely touched.

**Why does this matter for PCA?** PCA eigendecomposes the data's covariance matrix and then ranks the directions by eigenvalue, largest first: the direction with the largest eigenvalue is the one along which the data is "most spread out" and carries the most information (the first principal component). If the largest eigenvalue is far bigger than the second, the data varies almost entirely along one direction, and squeezing it onto that one direction loses almost no information — that's why "dimensionality reduction" works. Conversely, if all the eigenvalues are about the same size, every direction matters equally, and compression throws away a fair amount of information.

That's also why the last line of the code says "the further apart the two eigenvalues, the faster power iteration converges". Power iteration (an algorithm that finds the largest eigenvalue/eigenvector using nothing but repeated matrix multiplication) keeps multiplying a vector by A; each multiplication shrinks the non-dominant components relative to the dominant one, and the bigger the ratio, the faster they shrink, so the faster it converges to the dominant direction.


## Parameter experiment: both sides of the stability boundary

Build a matrix with eigenvalues 0.9 and 1.1, compute A^50, and watch the long-run behavior of the two directions:
- `0.9^50 ≈ 0.005`: that eigen-direction has decayed to nearly zero (stable)
- `1.1^50 ≈ 117`: that eigen-direction blows up exponentially (unstable)

Just checking whether the eigenvalue magnitudes cross the line at 1.0 is enough to predict the system's long-run behavior. Aurora's state space models guarantee stability by keeping the magnitudes of all the system matrix's eigenvalues inside the unit circle.

### Why is the criterion "magnitude less than 1" and not "value less than 1"?

In the example above, 0.9 and 1.1 are both positive, so comparing their values and comparing their "magnitudes" (absolute values) looks the same. But what if an eigenvalue is **negative**? Say one eigenvalue of the system matrix is λ=-0.8 — it satisfies "λ<1" (-0.8 really is less than 1). If the criterion were wrongly written as "λ<1 means stable", would anything go wrong?

The answer depends on how `λⁿ` actually behaves. A negative number raised to an even power is positive, and to an odd power is negative — the sign keeps alternating (this is called "oscillation"), but as long as |λ|<1, each swing is a little smaller:

- λ=-0.8: `(-0.8)¹=-0.8`, `(-0.8)²=0.64`, `(-0.8)³=-0.512`, ... the size keeps shrinking and tends to 0 (convergence — it just "zigzags toward 0" rather than falling steadily)
- But if λ=-1.5 (magnitude greater than 1): `(-1.5)¹=-1.5`, `(-1.5)²=2.25`, `(-1.5)³=-3.375`, ... the size keeps growing — oscillating again, but this time **oscillating outward and diverging**

So the condition "λ<1" is useless for negative eigenvalues (-1.5 satisfies λ<1 too, yet the system diverges)! What really decides convergence or divergence is only how far the number is from 0 — its **magnitude (absolute value)**: `|λ|<1` converges, `|λ|>1` diverges, whether it's positive or negative, real or complex (the "magnitude" of a complex number is `√(real² + imag²)`, and the same logic holds).


In [ ]:
# See for yourself: powers of a negative eigenvalue oscillate; only the magnitude predicts convergence or divergence
for lam in [-0.8, -1.5]:
    powers = [lam**n for n in range(1, 6)]
    trend = 'converges' if abs(lam) < 1 else 'diverges'
    print(f'λ={lam}: λ¹..λ⁵ = {[round(p, 3) for p in powers]}   |λ|={abs(lam)}  → {trend}')


## Wrap-up

You can now use `np.linalg.eig(A)` to get the array of eigenvalues and the eigenvector matrix `P`, and use `char_poly(A, lam)` to check whether any given `λ` is a true eigenvalue. This is the stability check for Aurora's state space models: eigendecompose the system matrix, and if every `|λ| < 1`, the recursive filter converges. The next section (**L18**) fills in the invertibility criterion — when P is invertible, so that `A = PDP⁻¹` reconstructs A exactly.

---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: eigendecomposition by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: A = [[4, 1], [2, 3]]. Work out the characteristic equation `det(A - λI) = 0` by hand and find the two eigenvalues.

**Question 2**: Check: for the larger eigenvalue λ₁ from Question 1, what should `char_poly(A, λ₁)` equal?

> 💡 This question may look like copying out the definition again — "of course a true eigenvalue plugged into char_poly gives 0". But the point isn't to repeat the derivation; it's to **run your own char_poly code on a concrete number yourself**. This is a self-check habit programmers and scientists use all the time: after writing a function, feed it an input whose answer you already know and confirm the code really computes it, rather than taking it for granted that the reasoning was sound. If char_poly has a bug (a flipped sign, a missing minus), this question is where you'll catch it first.

**Question 3**: If A = P diag(2, 5) P⁻¹, then A³ = P diag(?, ?) P⁻¹? (You don't need to know what P is.)

**Question 4**: A linear system's system matrix has eigenvalues [0.9, −0.8]. Is the system stable?  
(Criterion: stable if every |λᵢ| < 1 — note that −0.8 is negative; don't just look at its value, look at its magnitude.)

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

A = np.array([[4., 1.], [2., 3.]])

# Q1: eigenvalues
vals_ref = np.sort(np.linalg.eigvals(A))
assert np.allclose(np.sort(vals_ref), [2., 5.], atol=1e-10)
print(f"Q1 ✅  eigenvalues of A=[[4,1],[2,3]] = {np.sort(vals_ref)}  (by hand: λ²-7λ+10=0 → 5,2)")

# Q2: char_poly(A, λ₁) ≈ 0
lam1 = 5.0
try:
    cp = char_poly(A, lam1)
    assert abs(cp) < 1e-8, f"char_poly(A, 5) should be ≈0, got {cp}"
    print(f"Q2 ✅  char_poly(A, 5) = {cp:.2e}  (λ=5 is a true eigenvalue)")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement char_poly() first, then run this answer-check cell")

# Q3: A³ = P diag(2³,5³) P⁻¹
lam3_expected = [2**3, 5**3]  # = [8, 125]
vals3, vecs3 = np.linalg.eig(A)
D3 = np.diag(vals3**3)
A3_diag = vecs3 @ D3 @ np.linalg.inv(vecs3)
A3_ref  = np.linalg.matrix_power(A.astype(int), 3).astype(float)
assert np.allclose(A3_diag, A3_ref, atol=1e-8)
print(f"Q3 ✅  A³ = P·diag({lam3_expected[0]},{lam3_expected[1]})·P⁻¹  (=diag(2³,5³))")

# Q4: stability (every |λ|<1)
eigs_sys = [0.9, -0.8]
stable = all(abs(e) < 1 for e in eigs_sys)
assert stable
print(f"Q4 ✅  eigenvalues={eigs_sys}, |0.9|<1 and |-0.8|<1 → the system is stable (Aⁿ→0)")
print("\n🎉 Eigendecomposition whiteboard challenge passed! The idea of changing coordinates to make multiplication simple has sunk in.")

In [ ]:
# ✏️ Self-review for this lesson
l17_review = {
    "char_poly_implemented":       None,  # char_poly implemented and passes its asserts? True/False
    "eigendecomp_A_eq_PDP_inv":   None,  # understand what A=PDP⁻¹ means? True/False
    "matrix_power_via_diag":      None,  # know that Aⁿ=PDⁿP⁻¹? True/False
    "stability_criterion":         None,  # understand |λ|<1 → stable? True/False
    "whiteboard_passed":           None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l17_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l17_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L17 complete! On to L18: invertibility and rank')

---

→ **Next lesson** · [L18 · Invertibility and rank](L18_invertibility.ipynb)

> Next lesson covers **invertibility and rank**: rank = information, null space = the directions that get crushed, and diagnosing singular matrices.